<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Lab's Diagrams

Every diagram in `docs/figures/` whose name starts `diagram_` is drawn here, as plain SVG written by Python with the
standard library alone, so that it needs no tool beyond the notebook's own pins (`notebooks/requirements.txt`) and
diffs as text. The drawing is deterministic: integer coordinates, a fixed element order, no timestamps or random
identifiers, so running this notebook again on an unchanged tree changes no byte of any figure. The last cell checks
that, by drawing every diagram twice and comparing the bytes.

Run with `jupyter nbconvert --to notebook --execute --inplace notebooks/figures.ipynb`.


In [1]:
import os
from html import escape

# nbconvert runs a notebook from its own directory; the figures live under the repository's root.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
FIGURES = "docs/figures"
FONT = "DejaVu Sans, Verdana, Arial, sans-serif"

# One palette for every diagram: a repository or a role keeps its colour from figure to figure.
PALETTE = {
    "world": ("#e8f1fb", "#2f6db3"),   # Master Control, the world server
    "wire": ("#eef7ee", "#3d8b40"),    # Link, the wire
    "grid": ("#fdf1e3", "#c46f12"),    # tron-grid-lite, the Grid
    "brain": ("#f6ecfa", "#8a3fa8"),   # a Program, a creature's brain
    "lab": ("#fff8d9", "#9a7b00"),     # iso-lab's own tools and documents
    "data": ("#eef0f2", "#4d5a66"),    # data from outside: the real worms
    "note": ("#ffffff", "#7a7a7a"),    # a plain note
}


class Svg:
    """A tiny SVG writer: rounded boxes, arrows and text, every number an integer."""

    def __init__(self, width, height, title):
        self.width, self.height, self.title = width, height, title
        self.parts = []

    def box(self, x, y, w, h, title, lines=(), role="note", dashed=False):
        fill, stroke = PALETTE[role]
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        self.parts.append(
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="10" fill="{fill}" stroke="{stroke}" stroke-width="2"{dash}/>'
        )
        cx = x + w // 2
        ty = y + 26 if lines else y + h // 2 + 6
        self.parts.append(
            f'<text x="{cx}" y="{ty}" text-anchor="middle" font-size="16" font-weight="bold" fill="#1d1d1d">{escape(title)}</text>'
        )
        for i, line in enumerate(lines):
            self.parts.append(
                f'<text x="{cx}" y="{ty + 22 + 19 * i}" text-anchor="middle" font-size="13" fill="#333333">{escape(line)}</text>'
            )

    def arrow(self, points, label=None, label_at=None, dashed=False, both=False, colour="#444444"):
        d = "M " + " L ".join(f"{x} {y}" for x, y in points)
        dash = ' stroke-dasharray="6 4"' if dashed else ""
        start = ' marker-start="url(#head)"' if both else ""
        self.parts.append(
            f'<path d="{d}" fill="none" stroke="{colour}" stroke-width="2"{dash}{start} marker-end="url(#head)"/>'
        )
        if label:
            lx, ly = label_at
            self.parts.append(
                f'<text x="{lx}" y="{ly}" text-anchor="middle" font-size="13" font-style="italic" fill="#333333">{escape(label)}</text>'
            )

    def text(self, x, y, s, size=13, anchor="middle", bold=False, colour="#333333"):
        weight = ' font-weight="bold"' if bold else ""
        self.parts.append(
            f'<text x="{x}" y="{y}" text-anchor="{anchor}" font-size="{size}"{weight} fill="{colour}">{escape(s)}</text>'
        )

    def render(self):
        head = (
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{self.width}" height="{self.height}" '
            f'viewBox="0 0 {self.width} {self.height}" font-family="{FONT}">\n'
            f"<title>{escape(self.title)}</title>\n"
            '<defs><marker id="head" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" '
            'orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#444444"/></marker></defs>\n'
            f'<rect x="0" y="0" width="{self.width}" height="{self.height}" fill="#ffffff"/>\n'
        )
        return head + "\n".join(self.parts) + "\n</svg>\n"


def save(name, svg):
    path = os.path.join(FIGURES, name)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(svg.render())
    return path


DIAGRAMS = {}


def diagram(name):
    def register(draw):
        DIAGRAMS[name] = draw
        return draw
    return register


## How the Pieces Fit

The organisation's repositories and the lab's own tools, and what flows between them. A creature's brain - a Program -
sees the world only through the senses the Grid renders for its body, and acts only through that body; the world
itself is Master Control's, reached over Link. What a life leaves behind - the Disk and the log - and the recordings
of real worms meet in the lab's tools, where the benchmark compares them. A dashed box or arrow is planned, not built.


In [2]:
@diagram("diagram_pieces.svg")
def pieces():
    s = Svg(1100, 620, "How the pieces fit")
    s.text(550, 34, "How the pieces fit", size=22, bold=True, colour="#1d1d1d")
    s.text(550, 58, "a creature lives on the Grid; the lab compares its life with real worms")

    # The world, left to right: Master Control - Link - the Grid - a Program.
    s.box(30, 90, 220, 120, "Master Control", ["the world server: physics,", "food, the tick, the logs"], role="world")
    s.box(320, 110, 150, 80, "Link", ["the wire"], role="wire")
    s.box(540, 90, 230, 120, "The Grid", ["renders each body's senses", "and applies its actions"], role="grid")
    s.box(860, 90, 210, 120, "A Program", ["a creature's brain,", "e.g. the placeholder worm"], role="brain")
    s.arrow([(250, 150), (318, 150)], both=True)
    s.arrow([(470, 150), (538, 150)], both=True)
    s.arrow([(770, 125), (858, 125)], "senses", (814, 115))
    s.arrow([(858, 175), (772, 175)], "actions", (814, 198))

    # What a life leaves behind.
    s.box(30, 300, 220, 100, "Disk and log", ["a life, recorded"], role="world")
    s.arrow([(140, 210), (140, 298)], "writes", (172, 260))
    s.box(30, 480, 220, 100, "Clu", ["replays a log and checks", "it gives the same world"], role="world")
    s.arrow([(140, 400), (140, 478)], "replay", (170, 444))

    # The real worms and the lab's tools.
    s.box(860, 300, 210, 100, "Real worms", ["recordings of C. elegans", "(Zenodo, public)"], role="data")
    s.box(540, 300, 230, 100, "fetch, features", ["the lab's Rust tools:", "download, then measure"], role="lab")
    s.arrow([(858, 350), (772, 350)], "download", (814, 340))
    s.arrow([(250, 350), (538, 350)], "read the Disk", (394, 340), dashed=True)
    s.text(394, 372, "(Disk reader planned)", size=12, colour="#7a7a7a")
    s.box(540, 480, 230, 100, "The benchmark", ["is the Grid worm's behaviour", "within the real worms'?"], role="lab")
    s.arrow([(655, 400), (655, 478)], "both sets of measures", (740, 444))
    return s


## Drawing, and the Check That Nothing Moves

Every diagram is drawn and written to `docs/figures/`, then drawn again and compared byte for byte with what was
written. A difference would mean the drawing depends on something other than this notebook's code.


In [3]:
written = {name: save(name, draw()) for name, draw in sorted(DIAGRAMS.items())}
for name, path in sorted(written.items()):
    with open(path, encoding="utf-8") as f:
        on_disk = f.read()
    assert on_disk == DIAGRAMS[name]().render(), f"{name} is not deterministic"
    print(f"{path}: {len(on_disk.encode('utf-8'))} bytes, redrawn identically")


docs/figures/diagram_pieces.svg: 6175 bytes, redrawn identically
